# MiniMax H3 (ComfyUI) Colab ノートブック検証用

CLI の `A100` 割り当て混雑時の確認や、フェーズ1実機ベンチマーク・`workflow_api.json` 書き出しを行うためのセルフコンテインド・ノートブックです。

### 手順
1. メニュー **「ランタイム ＞ ランタイムのタイプを変更」** で **A100 GPU** を選択して接続します。
2. セル 1〜4 を上から順に実行し、表示されたリンクから ComfyUI GUI を開きます（`video_minimax_h3_t2v.json` が自動ロードされ、Dev Mode も ON です）。
3. `960x544 / 5秒 / 8step` で 2 本連続生成して 1本目・2本目の所要時間を記録し、**Workflow → Export (API)** から `workflow_api.json` を保存します。
4. 終了時は必ず **セル 6 (`drive.flush_and_unmount()`)** を実行してから、ランタイムを接続解除して削除してください。


In [2]:
import shutil
import subprocess

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU が割り当てられていません (CPU ランタイム)。ランタイムのタイプを A100 へ変更してください。")

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_info)
if "A100" not in gpu_info and "G4" not in gpu_info and "H100" not in gpu_info:
    print("[WARN] A100/G4/H100 以外のGPUです。MiniMax H3 (~42GB) は低速またはメモリ不足の可能性があります。")

print(subprocess.check_output(["free", "-h"], text=True).strip())
print(subprocess.check_output(["df", "-h", "/"], text=True).strip())

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
total        used        free      shared  buff/cache   available
Mem:            83Gi       1.9Gi        79Gi       2.4Mi       3.1Gi        81Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   83M  236G   1% /


In [3]:
from pathlib import Path

from google.colab import drive

DRIVE_ROOT = "ComfyUI"
REQUIRED_ASSETS = [
    "models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "models/vae/minimax_h3_video_vae_int8_convrot.safetensors",
    "models/vae/minimax_h3_audio_vae_fp32.safetensors",
    "models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
]

drive.mount("/content/drive")
drive_base = Path("/content/drive/MyDrive") / DRIVE_ROOT
if not drive_base.is_dir():
    raise FileNotFoundError(f"Drive root が見つかりません: {drive_base}")

missing = []
for rel in REQUIRED_ASSETS:
    target = drive_base / rel
    if not target.is_file():
        missing.append(str(target))
    else:
        size_gb = target.stat().st_size / (1024**3)
        print(f"[OK] {rel} ({size_gb:.2f} GiB)")

if missing:
    raise FileNotFoundError("不足しているモデル資産があります:\n" + "\n".join(missing))
print("全必須モデル 5 ファイルの確認が完了しました。")

Mounted at /content/drive
[OK] models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors (19.53 GiB)
[OK] models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors (14.61 GiB)
[OK] models/vae/minimax_h3_video_vae_int8_convrot.safetensors (2.62 GiB)
[OK] models/vae/minimax_h3_audio_vae_fp32.safetensors (0.56 GiB)
[OK] models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors (1.82 GiB)
全必須モデル 5 ファイルの確認が完了しました。


In [4]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import urllib.request

INSTALL_SAGEATTENTION = False
DRIVE_ROOT = "ComfyUI"
COMFY_DIR = Path("/content/ComfyUI")
DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_ROOT

WORKFLOW_URLS = [
    "https://raw.githubusercontent.com/xzyozi/colab-cli-comfyui-runner/develop/workflows/minimax_h3/video_minimax_h3_t2v.json",
    "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json",
]

AUTOLOAD_INIT = """WEB_DIRECTORY = "./js"
NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS", "WEB_DIRECTORY"]
"""

AUTOLOAD_JS = """import { app } from "../../scripts/app.js";

app.registerExtension({
  name: "ccr.autoload.minimax_h3",
  async setup() {
    setTimeout(async () => {
      try {
        if (sessionStorage.getItem("ccr_autoloaded") === "1") {
          return;
        }
        const wfUrl = new URL("./video_minimax_h3_t2v.json", import.meta.url);
        const resp = await fetch(wfUrl, { cache: "no-store" });
        if (!resp.ok) {
          return;
        }
        const wf = await resp.json();
        await app.loadGraphData(wf);
        sessionStorage.setItem("ccr_autoloaded", "1");
      } catch (err) {
        console.warn("[ccr_autoload] error:", err);
      }
    }, 600);
  },
});
"""

if not shutil.which("uv"):
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
if not shutil.which("comfy"):
    subprocess.run(["uv", "pip", "install", "--system", "-q", "comfy-cli"], check=True)

if not COMFY_DIR.is_dir():
    res = subprocess.run(
        ["comfy", "--skip-prompt", "--workspace", str(COMFY_DIR), "install", "--nvidia", "--fast-deps"],
        check=False,
    )
    if res.returncode != 0:
        shutil.rmtree(COMFY_DIR, ignore_errors=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)],
            check=True,
        )
        subprocess.run(
            ["uv", "pip", "install", "--system", "-q", "-r", str(COMFY_DIR / "requirements.txt")],
            check=True,
        )

if INSTALL_SAGEATTENTION:
    subprocess.run(["uv", "pip", "install", "--system", "-q", "sageattention"], check=True)

for subdir in ("diffusion_models", "text_encoders", "vae", "loras", "embeddings"):
    src = DRIVE_BASE / "models" / subdir
    dst = COMFY_DIR / "models" / subdir
    if src.is_dir():
        if dst.is_symlink() or dst.exists():
            shutil.rmtree(dst) if dst.is_dir() and not dst.is_symlink() else dst.unlink()
        os.symlink(src, dst)

(DRIVE_BASE / "output").mkdir(parents=True, exist_ok=True)
out_dst = COMFY_DIR / "output"
if out_dst.is_symlink() or out_dst.exists():
    shutil.rmtree(out_dst) if out_dst.is_dir() and not out_dst.is_symlink() else out_dst.unlink()
os.symlink(DRIVE_BASE / "output", out_dst)

if (DRIVE_BASE / "input").is_dir():
    (COMFY_DIR / "input").mkdir(parents=True, exist_ok=True)
    in_dst = COMFY_DIR / "input" / "drive_assets"
    if in_dst.is_symlink() or in_dst.exists():
        shutil.rmtree(in_dst) if in_dst.is_dir() and not in_dst.is_symlink() else in_dst.unlink()
    os.symlink(DRIVE_BASE / "input", in_dst)

wf_bytes = None
for url in WORKFLOW_URLS:
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            wf_bytes = resp.read()
        print(f"Downloaded workflow template from: {url}")
        break
    except Exception as exc:
        print(f"Fetch failed ({url}): {exc}")
if wf_bytes is None:
    raise RuntimeError("公式ワークフローテンプレートの取得に失敗しました。")

user_wf_dir = COMFY_DIR / "user" / "default" / "workflows"
user_wf_dir.mkdir(parents=True, exist_ok=True)
(user_wf_dir / "video_minimax_h3_t2v.json").write_bytes(wf_bytes)

settings_path = COMFY_DIR / "user" / "default" / "comfy.settings.json"
settings_path.write_text(json.dumps({"Comfy.DevMode": True}, indent=2) + "\n", encoding="utf-8")

ext_dir = COMFY_DIR / "custom_nodes" / "ccr_autoload"
js_dir = ext_dir / "js"
js_dir.mkdir(parents=True, exist_ok=True)
(ext_dir / "__init__.py").write_text(AUTOLOAD_INIT, encoding="utf-8")
(js_dir / "autoload.js").write_text(AUTOLOAD_JS, encoding="utf-8")
(js_dir / "video_minimax_h3_t2v.json").write_bytes(wf_bytes)
print("ComfyUI 環境・Drive リンク・ワークフロー自動ロード設定が完了しました。")

Downloaded workflow template from: https://raw.githubusercontent.com/xzyozi/colab-cli-comfyui-runner/develop/workflows/minimax_h3/video_minimax_h3_t2v.json
ComfyUI 環境・Drive リンク・ワークフロー自動ロード設定が完了しました。


In [5]:
from pathlib import Path
import subprocess
import sys
import time
import urllib.request

from google.colab import output

COMFY_PORT = 8188
LOG_PATH = Path("/tmp/comfyui.log")

pgrep = subprocess.run(["pgrep", "-f", f"[m]ain.py.*--port {COMFY_PORT}"], capture_output=True, text=True)
if pgrep.returncode != 0:
    with LOG_PATH.open("wb") as log_file:
        subprocess.Popen(
            [
                sys.executable,
                "/content/ComfyUI/main.py",
                "--listen",
                "127.0.0.1",
                "--port",
                str(COMFY_PORT),
                "--enable-cors-header",
            ],
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

ready = False
for _ in range(36):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=3) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(5)

if not ready:
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:])
    raise RuntimeError("ComfyUI が 180 秒以内に起動しませんでした。")

print("ComfyUI 起動完了。以下のリンクから GUI を開いてください（ワークフロー自動ロード & DevMode ON）：")
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({COMFY_PORT})")
print("Proxy URL:", proxy_url)
output.serve_kernel_port_as_window(COMFY_PORT)

ComfyUI 起動完了。以下のリンクから GUI を開いてください（ワークフロー自動ロード & DevMode ON）：
Proxy URL: https://8188-gpu-a100-s-usc1b0-1vt5ltfaphg34-b.us-central1.prod.r.colab.dev
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [10]:
from pathlib import Path

DRIVE_ROOT = "ComfyUI"
out_dir = Path("/content/drive/MyDrive") / DRIVE_ROOT / "output"
print("=== Output files ===")
for item in sorted(out_dir.rglob("*")):
    if item.is_file():
        print(f"{item.relative_to(out_dir)} ({item.stat().st_size / (1024**2):.2f} MiB)")

log_path = Path("/tmp/comfyui.log")
if log_path.is_file():
    print("\n=== ComfyUI log (last 40 lines) ===")
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-40:]))

=== Output files ===
video/MiniMax_H3_00001_.mp4 (1.47 MiB)

=== ComfyUI log (last 40 lines) ===
[INFO] Detected mixed precision quantization
[INFO] Using mixed precision operations
[INFO] Native ops: int8_tensorwise, convrot_w4a4, asym_w4a8_int8, w6a8_int8 , emulated ops: mxfp8, float8_e4m3fn, nvfp4, float8_e5m2
[INFO] model weight dtype torch.bfloat16, manual cast: torch.bfloat16
[INFO] model_type FLOW_AV
[INFO] Model storage policy: fast_disk=False paths=['/content/drive/MyDrive/ComfyUI/models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors']
[INFO] Requested to load MiniMaxH3
[INFO] Model MiniMaxH3 prepared for dynamic VRAM loading. 19995MB Staged. 0 patches attached. Force pre-loaded 210 weights: 1175 KB.

100%|██████████| 20/20 [01:09<00:00,  3.49s/it]
[INFO] Comfy model compiler graph breaks: 1, rogues: 0
[INFO] Requested to load MiniMaxH3AudioVAE
[INFO] Model MiniMaxH3AudioVAE prepared for dynamic VRAM loading. 576MB Staged. 0 patches attached. Force pre-loade

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
print("DriveFS フラッシュ・アンマウント完了。Colab メニューからランタイムを削除して課金を停止してください。")

In [12]:
from pathlib import Path
import subprocess

print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv"],
    text=True,
))
print(subprocess.check_output(["free", "-h"], text=True))
ps = subprocess.run(["pgrep", "-af", "[m]ain.py"], capture_output=True, text=True).stdout.strip()
print("ComfyUI proc:", ps)
if ps:
    pid = ps.split()[0]
    print(subprocess.run(["ps", "-p", pid, "-o", "pid,pcpu,pmem,rss,stat,etime"], capture_output=True, text=True).stdout)
    lines = Path("/tmp/comfyui.log").read_text(encoding="utf-8", errors="replace").splitlines()
    print("--- /tmp/comfyui.log (last 35 lines) ---")
    print("\n".join(lines[-35:]))

utilization.gpu [%], memory.used [MiB], memory.total [MiB]
100 %, 40056 MiB, 40960 MiB

               total        used        free      shared  buff/cache   available
Mem:            83Gi        43Gi       1.4Gi        21Mi        39Gi        40Gi
Swap:             0B          0B          0B

ComfyUI proc: 19127 /usr/bin/python3 /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --enable-cors-header
    PID %CPU %MEM   RSS STAT     ELAPSED
  19127 48.5 50.4 44131756 Ssl     05:32

--- /tmp/comfyui.log (last 35 lines) ---
[INFO] VAE load device: cuda:0, offload device: cpu, dtype: torch.float16
[INFO] Found quantization metadata version 1
[INFO] Using MixedPrecisionOps for text encoder
[INFO] Model storage policy: fast_disk=False paths=['/content/drive/MyDrive/ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors']
[INFO] CLIP/text encoder model load device: cuda:0, offload device: cpu, current: cpu, dtype: torch.float16
[INFO] Found quantization metadata vers

In [11]:
from pathlib import Path
import subprocess
import sys
import time
import urllib.request

from google.colab import output

# 任意: civitai.red の API トークンを入れると H3 用 LoRA も Drive に自動保存します
CIVITAI_TOKEN = ""
DOWNLOAD_LORAS = {
    # version_id: 保存ファイル名 (必要なものだけ残す/コメント解除)
    3449495: "H3_VBVR_Pro-V2.safetensors",        # VBVR Reasoning Pro V2 (~1.16 GB)
    3405852: "H3_DR34ML4Y_P1.safetensors",        # DR34ML4Y NSFW AIO (~564 MB)
    # 3478361: "M3_Unlocked_V3.1_B.safetensors",  # M3 Unlocked V3.1_B (~575 MB)
}

COMFY_DIR = Path("/content/ComfyUI")
DRIVE_LORAS = Path("/content/drive/MyDrive/ComfyUI/models/loras")
BRANCH_RAW = "https://raw.githubusercontent.com/xzyozi/colab-cli-comfyui-runner/feat/minimax-h3-phase1"

for repo_url in [
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/rgthree/rgthree-comfy.git",
]:
    name = repo_url.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git")
    dst = COMFY_DIR / "custom_nodes" / name
    if not dst.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(dst)], check=True)
    req = dst / "requirements.txt"
    if req.is_file():
        subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(req)], check=True)

wf_url = f"{BRANCH_RAW}/workflows/minimax_h3/video_minimax_h3_i2v.json"
with urllib.request.urlopen(wf_url, timeout=30) as resp:
    wf_bytes = resp.read()

(COMFY_DIR / "user/default/workflows").mkdir(parents=True, exist_ok=True)
(COMFY_DIR / "user/default/workflows/video_minimax_h3_i2v.json").write_bytes(wf_bytes)
js_dir = COMFY_DIR / "custom_nodes/ccr_autoload/js"
js_dir.mkdir(parents=True, exist_ok=True)
(js_dir / "video_minimax_h3_i2v.json").write_bytes(wf_bytes)
(js_dir / "autoload.js").write_text(
    """import { app } from "../../scripts/app.js";
app.registerExtension({
  name: "ccr.autoload.minimax_h3",
  async setup() {
    setTimeout(async () => {
      try {
        if (sessionStorage.getItem("ccr_autoloaded_i2v") === "1") return;
        const resp = await fetch(new URL("./video_minimax_h3_i2v.json", import.meta.url), { cache: "no-store" });
        if (!resp.ok) return;
        await app.loadGraphData(await resp.json());
        sessionStorage.setItem("ccr_autoloaded_i2v", "1");
      } catch (err) {
        console.warn("[ccr_autoload] error:", err);
      }
    }, 600);
  },
});
""",
    encoding="utf-8",
)

if CIVITAI_TOKEN.strip():
    DRIVE_LORAS.mkdir(parents=True, exist_ok=True)
    for vid, fname in DOWNLOAD_LORAS.items():
        out_path = DRIVE_LORAS / fname
        if out_path.is_file():
            print(f"[SKIP] {fname} (already exists)")
            continue
        url = f"https://civitai.red/api/download/models/{vid}?token={CIVITAI_TOKEN.strip()}"
        print(f"Downloading {fname} ...")
        subprocess.run(["curl", "-L", "--fail", "-o", str(out_path), url], check=True)

pgrep = subprocess.run(["pgrep", "-f", "[m]ain.py.*--port 8188"], capture_output=True, text=True)
for pid in pgrep.stdout.split():
    subprocess.run(["kill", pid], check=False)
time.sleep(2)

with Path("/tmp/comfyui.log").open("wb") as log_file:
    subprocess.Popen(
        [sys.executable, "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188", "--enable-cors-header"],
        stdin=subprocess.DEVNULL,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

for _ in range(36):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("ComfyUI の再起動がタイムアウトしました。")

print("Proxy URL:", output.eval_js("google.colab.kernel.proxyPort(8188)"))
output.serve_kernel_port_as_window(8188)


Proxy URL: https://8188-gpu-a100-s-usc1b0-1vt5ltfaphg34-b.us-central1.prod.r.colab.dev
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>